# To do the conversion

This notebook declares the network so that we can run it and perform the float to fixed conversion automatically using the float2fixed library

In [ ]:
from pathlib import Path
from lava.magma.core.process.variable import Var

In [ ]:
import json
import numpy as np

def serialize_dict(data_dict):
    def convert(obj):
        if isinstance(obj, np.ndarray):
            return {'data': obj.tolist(), 'dtype': str(obj.dtype)}
        if isinstance(obj, (np.int64, np.float32)):
            return obj.item()
        raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

    return json.dumps(data_dict, default=convert)


def deserialize_dict(json_str):
    def convert(obj):
        if isinstance(obj, dict) and 'data' in obj and 'dtype' in obj:
            return np.array(obj['data'], dtype=np.dtype(obj['dtype']))
        if isinstance(obj, list):
            return np.array(obj)
        return obj

    return json.loads(json_str, object_hook=lambda d: {k: convert(v) for k, v in d.items()})

def update_dict_with_objects(obj_list, dict_to_update):
    for obj in obj_list:
        if obj.id in dict_to_update:
            dict_to_update[obj.name] = dict_to_update.pop(obj.id)

def to_dict(obj,obj_name):
    obj_dict = {}
    for attr in dir(obj):
        if not callable(getattr(obj, attr)) and not attr.startswith("__") and not attr.startswith("_"):
            if type(getattr(obj, attr)) == Var:
                print(attr)
                obj_dict[attr] = getattr(obj, attr).init
    return_dict = {obj_name: obj_dict}
    return return_dict

In [ ]:
from lava.proc.lif.process import LIF
from lava.proc.dense.process import Dense
from lava.utils.weightutils import SignMode

network_name = "../data/lava_inhibitory_best.json"
#path = f"{Path.home()}/snntorch_network/notebook/Trained/network_best.npz"
path = f"{Path.home()}/snntorch_network/nni_experiments/inibitory_lif_no_encoder_best/results/ot6eqima/trials/oqxoS/Trained/network_best.npz"
#path = f"{Path.home()}/snntorch_network/nni_experiments/inibitory_lif_no_encoder_worst/results/ipk2erm5/trials/whXvC/Trained/network_best.npz"
#path = f"{Path.home()}/snntorch_network/nni_experiments/inibitory_lif_no_encoder_balanced/results/4m8j0yfa/trials/yAnF7/Trained/network_best.npz"
data = np.load(path,allow_pickle=True)

# In Lava, if we pass the quantized weights but the floating states, things
# break. To avoid this issues, we use the floating point weights and states,
# to quantize them using the automatic converter.

# In_Dense
linear1_w= data['linear1']
# In_LIF
leaky1_vth= data['leaky1_vth']
# Invert it to make it compatible with Lava
leaky1_betas= 1-data['leaky1_betas']
# On snntorch sometimes the decay is larger than 1, which results in a negative
# beta. This is not valid, so we set it to 0
leaky1_betas= leaky1_betas if leaky1_betas >= 0 else np.zeros(leaky1_betas.shape)
print(f"leaky1_betas: {leaky1_betas}")
print(f"leaky1_vth: {leaky1_vth}")

# In-R_Dense
linear2_w = data['linear2']
# F_LIF
leaky2_vth= data['recurrent_vth']
leaky2_betas= 1 - data['recurrent_betas']
leaky2_betas= leaky2_betas if  leaky2_betas >= 0 else np.zeros(leaky2_betas.shape)
print(f"leaky2_betas: {leaky2_betas}")
print(f"leaky2_vth: {leaky2_vth}")

# Recurrent
recurrent_in_weights = data['input_dense']     # Input dense
recurrent_out_weights = - data['output_dense'] # Output dense. Negative because it is inhibitory
# ILIF
recurrent_vth = data['activation_vth']
recurrent_leaky_betas = 1 - data['activation_betas']
recurrent_leaky_betas= recurrent_leaky_betas if recurrent_leaky_betas >= 0 else np.zeros(recurrent_leaky_betas.shape)
print(f"recurrent_leaky_betas: {recurrent_leaky_betas}")
print(f"recurrent_vth: {recurrent_vth}")

# Out_Dense
linear3_w = data['linear3']
# O_LIF
leaky3_vth= data['leaky2_vth']
leaky3_betas= 1 - data['leaky2_betas']
leaky3_betas= leaky3_betas if leaky3_betas >= 0 else np.zeros(leaky3_betas.shape)
print(f"leaky3_betas: {leaky3_betas}")
print(f"leaky3_vth: {leaky3_vth}")

'''
       +-----------+     +-----------+    +------------+              +-----------+          +-----------+     +-----------+
 +--+  |           |     |           |    |            |  +--+        |           |          |           |     |           |    +--+
 |i |->|  In_Dense |---->|  In_LIF   |--->| In-R_Dense |--| -|------->|  F_LIF    |-------+->| Out_Dense |---->|  O_LIF    |--->|o |
 +--+  |           |     |           |    |            |  +--+        |           |       |  |           |     |           |    +--+
       +-----------+     +-----------+    +------------+   A          +-----------+       |  +-----------+     +-----------+
                                                           |                              |
                                                           |   +------+ +------+ +------+ |
                                                           |   |      | |      | |      | |
                                                           +---|Dense |<| ILIF |<|Dense |<+
                                                               |      | |      | |      |
                                                               +------+ +------+ +------+
No need for delays. The network is robust enough to work without them.
'''

linear1 = Dense(weights=linear1_w, num_message_bits=32, name="linear1")

leaky1 = LIF(shape=(linear1_w.shape[0],),
                    u = np.zeros(linear1_w.shape[0]),
                    v = np.zeros(linear1_w.shape[0]),
                    du = 1.0,
                    dv = leaky1_betas,
                    vth=leaky1_vth,
                    log_config=0,
                    name= "leaky1"
                )

linear1.a_out.connect(leaky1.a_in)

# TODO: check if SignMode.MIXED is necessary
linear2 = Dense(weights=linear2_w, num_message_bits=0, sign_mode=SignMode.MIXED, name="linear2")
linear2.s_in.connect_from(leaky1.s_out)

leaky2 = LIF(shape=(linear2_w.shape[0],),
                    u = np.zeros(linear2_w.shape[0]),
                    v = np.zeros(linear2_w.shape[0]),
                    du = 1.0,
                    dv= leaky2_betas,
                    vth=leaky2_vth,
                    log_config=0,
                    name= "leaky2"
                )
#sum.a_out.connect(leaky2.a_in)
linear2.a_out.connect(leaky2.a_in)
#leaky2.a_in.connect_from(linear2.a_out)

recurrent_in = Dense(weights=recurrent_in_weights, num_message_bits=0, name="recurrent_in")
leaky2.s_out.connect(recurrent_in.s_in)

ahpc = LIF(shape=(recurrent_in_weights.shape[0],),
                    u = np.zeros(recurrent_in_weights.shape[0]),
                    v = np.zeros(recurrent_in_weights.shape[0]),
                    du = 1.0,
                    dv = recurrent_leaky_betas,
                    vth=recurrent_vth,
                    log_config=0,
                    name= "inibitory_leaky"
                )

recurrent_in.a_out.connect(ahpc.a_in)

recurrent_out = Dense(weights=recurrent_out_weights, num_message_bits=0, name="recurrent_out")
recurrent_out.s_in.connect_from(ahpc.s_out)
recurrent_out.a_out.connect(leaky2.a_in)

linear3 = Dense(weights=linear3_w, num_message_bits=0, name="linear3")
linear3.s_in.connect_from(leaky2.s_out)

leaky3 = LIF(shape=(linear3_w.shape[0],),
                    u = np.zeros(linear3_w.shape[0]),
                    v = np.zeros(linear3_w.shape[0]),
                    du = 1.0,
                    dv = leaky3_betas,
                    vth=leaky3_vth,
                    log_config=0,
                    name= "leaky3"
                )
leaky3.a_in.connect_from(linear3.a_out)

leaky1_betas: 0.8231102824211121
leaky1_vth: 1.924142837524414
leaky2_betas: 0.11010128259658813
leaky2_vth: 0.9596342444419861
recurrent_leaky_betas: 0.04187434911727905
recurrent_vth: 0.4355089068412781
leaky3_betas: 0.0
leaky3_vth: 0.791836142539978


In [ ]:

from lava.magma.core.run_conditions import RunSteps
from lava.magma.core.run_configs import Loihi1SimCfg, Loihi2SimCfg
from lava.utils.float2fixed import Float2FixedConverter # https://github.com/lava-nc/lava/blob/f2f_conv/src/lava/utils/float2fixed.py
import numpy as np
#from tqdm import tqdm

converter = Float2FixedConverter()
converter.set_run_cfg(fixed_pt_run_cfg=Loihi1SimCfg(select_tag='fixed_pt'),
                      floating_pt_run_cfg=Loihi1SimCfg(select_tag='floating_pt'))
linear1_dict = to_dict(linear1, 'linear1')
leaky1_dict = to_dict(leaky1, 'leaky1')

# List of processes to convert to fixed point
# We pass weights and states to the converter
converter.convert([leaky2, leaky3, ahpc, linear2, recurrent_in, recurrent_out, linear3], num_steps=200)

scaled_params_copy = converter.scaled_params.copy()

a_buff
num_message_bits
weights
bias_exp
bias_mant
du
dv
u
v
vth


In [5]:
linear1_dict

{'linear1': {'a_buff': 0,
  'num_message_bits': 32,
  'weights': array([[ 2.63966382e-01,  2.36511186e-01, -3.63286912e-01, -2.46764049e-01, -2.31889412e-01, -4.77008611e-01],
         [-2.85761915e-02,  6.77592382e-02,  2.51543403e-01, -2.64315516e-01, -4.18906868e-01, -3.70737493e-01],
         [ 1.23447970e-01,  3.49845290e-01, -3.01940084e-01,  3.30483377e-01,  8.01362842e-02,  8.39225054e-02],
         [ 8.69152620e-02,  7.90967569e-02,  2.13162646e-01, -2.52694696e-01, -2.60411888e-01,  3.70540880e-02],
         [ 2.79317945e-01,  4.76858050e-01,  1.97225243e-01, -2.87004173e-01,  2.58909672e-01,  1.64742693e-01],
         [ 1.09877311e-01, -8.63793269e-02,  7.80935064e-02,  1.47043362e-01, -4.77671117e-01, -4.53773618e-01],
         [ 2.62326375e-02, -2.98796952e-01, -1.39431953e-01,  3.44442397e-01, -4.77682710e-01, -1.70555890e-01],
         [-7.91777000e-02, -1.64973944e-01, -1.19711570e-01,  2.46561542e-01, -3.73652518e-01,  2.59476781e-01],
         [ 2.01091751e-01,  1.410

In [6]:
print(data["linear2_quant"])

[[ 17  21   5 ...   6 -16 -18]
 [-15   0  14 ... -10 -11  14]
 [ 14  23 -11 ...  -4 -22  -5]
 ...
 [  4  -9   1 ... -22 -10 -14]
 [ 12   4  -5 ...  16  -1 -10]
 [ -6  17   2 ...  21 -11   3]]


In [ ]:
# Save quantized version

obj_list = [leaky1, leaky2, leaky3, ahpc, linear1, linear2, linear3, recurrent_in, recurrent_out]

# Exchange number with name
update_dict_with_objects(obj_list, scaled_params_copy)
scaled_params_copy.update(linear1_dict)
scaled_params_copy.update(leaky1_dict)
scaled_params_copy['linear2']['weights'] = data["linear2_quant"]
print(scaled_params_copy)
for key, value in scaled_params_copy.items():
    print(f"\n{key}: {value}\n")
json_str = serialize_dict(scaled_params_copy)
with open(network_name, 'w') as json_file:
    json_file.write(json_str)


{'leaky2': {'bias_exp': 0, 'v': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0